In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/home/labib/Desktop/labib_codes/uiu_codes/7th_trimester/da_project/da_project_local/data/processed/olist_nlp_dataset.csv')
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,actual_delivery_days,delivery_delay_days,clean_review,review_length_words
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,8.0,-8.0,não testei o produto ainda mas ele veio corret...,32
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,13.0,-6.0,muito bom o produto,4
2,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,13.0,-13.0,o produto foi exatamente o que eu esperava e e...,20
3,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09,1.0,a1804276d9941ac0733cfd409f5206eb,...,e07549ef5311abcc92ba1784b093fb56,2.0,NaN,fiquei triste por n ter me atendido.,2017-05-13 00:00:00,2017-05-13 20:25:42,NaN,NaN,fiquei triste por n ter me atendido,7
4,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16 19:41:10,2017-05-16 19:50:18,2017-05-18 11:40:40,2017-05-29 11:18:31,2017-06-07,1.0,08574b074924071f4e201e151b152b4e,...,15898b543726a832d4137fbef5d1d00e,1.0,NaN,Aguardando retorno da loja,2017-05-30 00:00:00,2017-05-30 23:13:47,12.0,-9.0,aguardando retorno da loja,4


In [2]:
import pandas as pd

# 1. Load original dataset
df = pd.read_csv('/home/labib/Desktop/labib_codes/uiu_codes/7th_trimester/da_project/da_project_local/data/processed/olist_nlp_dataset.csv')

# 2. Select columns relevant to NLP and joining with ML teammate
nlp_cols = [
    'order_id',
    'review_score', 
    'review_comment_title', 
    'review_comment_message', 
    'clean_review'
]
df_nlp = df[nlp_cols].copy()

# 3. Handle missing values
# Fill missing titles with an empty string so we can concatenate
df_nlp['review_comment_title'] = df_nlp['review_comment_title'].fillna("")

# Combine title and message for a richer, single text feature
df_nlp['full_review_text'] = df_nlp['review_comment_title'] + " " + df_nlp['review_comment_message']
df_nlp['full_review_text'] = df_nlp['full_review_text'].str.strip()

# Drop rows where 'clean_review' is entirely missing
df_nlp = df_nlp.dropna(subset=['clean_review'])

# 4. Finalize dataset
final_cols = ['order_id', 'review_score', 'full_review_text', 'clean_review']
df_nlp_clean = df_nlp[final_cols]

# 5. Save to CSV for your experiments
df_nlp_clean.to_csv('/home/labib/Desktop/labib_codes/uiu_codes/7th_trimester/da_project/da_project_local/notebooks/nlp_evaluation/cleaned_nlp_dataset.csv', index=False)

In [3]:
import pandas as pd
import ollama
from tqdm import tqdm
import os
import time
import json

# 1. Configuration
INPUT_CSV = 'cleaned_nlp_dataset.csv'
OUTPUT_JSON = 'translated_nlp_dataset.jsonl' # Switched to .jsonl
MODEL_NAME = 'gpt-oss:20b'

# 2. Load the dataset
df = pd.read_csv(INPUT_CSV)

# 3. Define the translation prompt function
def translate_to_english(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    prompt = f"Translate the following Brazilian Portuguese e-commerce review into English. Provide ONLY the translation, without any explanations or introductory text:\n\n{text}"
    
    try:
        response = ollama.chat(model=MODEL_NAME, messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ])
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error translating text: {e}")
        time.sleep(5) # Wait for server to potentially recover
        return None

# 4. Process data with real-time saving to JSONL
translated_ids = set()
# If the JSONL file exists, read it to see what's already been translated.
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Use 'order_id' as the unique identifier
                translated_ids.add(json.loads(line)['order_id'])
            except (json.JSONDecodeError, KeyError):
                # Ignore malformed lines or lines missing the key
                continue

# Filter the dataframe to only include reviews that haven't been translated yet.
df_pending = df[~df['order_id'].isin(translated_ids)]
records_to_translate = df_pending.to_dict('records')

print(f"Total reviews: {len(df)}")
print(f"Already translated: {len(translated_ids)}")
print(f"Remaining to translate: {len(records_to_translate)}")

# 5. Process and append one by one to JSONL file
with open(OUTPUT_JSON, 'a', encoding='utf-8') as f:
    for record in tqdm(records_to_translate, desc="Translating and Saving to JSONL"):
        portuguese_text = record['full_review_text']
        
        translation = translate_to_english(portuguese_text)
        
        if translation is not None:
            record['english_translation'] = translation
            # Write the translated record as a new JSON line
            f.write(json.dumps(record) + '\n')

print("\nTranslation complete. Results saved to:", OUTPUT_JSON)


Total reviews: 44237
Already translated: 0
Remaining to translate: 44237


Translating and Saving to JSONL:   5%|▍         | 2118/44237 [1:18:31<21:39:00,  1.85s/it]

Error translating text: an error was encountered while running the model: CUDA error (status code: 500)


Translating and Saving to JSONL:  16%|█▌        | 6907/44237 [4:06:53<19:44:57,  1.90s/it]

Error translating text: an error was encountered while running the model: CUDA error (status code: 500)


Translating and Saving to JSONL:  59%|█████▉    | 26013/44237 [15:09:50<17:13:47,  3.40s/it]

Error translating text: an error was encountered while running the model: CUDA error: the launch timed out and was terminated
  current device: 0, in function ggml_backend_cuda_synchronize at /ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:2981
  cudaStreamSynchronize(cuda_ctx->stream())
/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:94: CUDA error (status code: 500)


Translating and Saving to JSONL: 100%|██████████| 44237/44237 [25:49:44<00:00,  2.10s/it]   


Translation complete. Results saved to: translated_nlp_dataset.jsonl


In [5]:
import pandas as pd

df_translated = pd.read_json('translated_nlp_dataset.jsonl', lines=True)
print(df_translated)

                               order_id  review_score  \
0      e481f51cbdc54678b7cc49136f2d6af7             4   
1      53cdb2fc8bc7dce0b6741e2150273451             4   
2      949d5b44dbf5de918fe9c16f97b45f8a             5   
3      136cce7faa42fdb2cefd53fdc79a6098             2   
4      e6ce16cb79ec1d90b1da9085a6118aeb             1   
...                                 ...           ...   
44229  b0f4af5c1b06e24fef510703bfe9f0a6             5   
44230  63943bddc261676b46f01ca7ac2f7bd8             4   
44231  83c1379a015df1e13d02aae0204711ab             5   
44232  11c177c8e97725db2631073c19f07b62             2   
44233  11c177c8e97725db2631073c19f07b62             2   

                                        full_review_text  \
0      Não testei o produto ainda, mas ele veio corre...   
1                  Muito boa a loja Muito bom o produto.   
2      O produto foi exatamente o que eu esperava e e...   
3                   fiquei triste por n ter me atendido.   
4              